# Anexo A. Reproducibilidad

Este anexo registra todo lo necesario para repetir los resultados: versiones de las librerías,
hardware, configuración de Spark, semillas, configuración del experimento y manifiesto de datos.

**Para reproducir de punta a punta** (ver `README.md`):

```bash
conda env create -f environment.yml && conda activate dl-ctr
# descargar train.gz de Kaggle en data/raw/
bash scripts/run_all.sh          # ejecuta los notebooks en orden y construye el Jupyter Book
```

In [1]:
import sys, json, platform, subprocess
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src") if Path.cwd().name == "notebooks" else str(Path.cwd() / "src"))
from importlib.metadata import version, PackageNotFoundError
import pandas as pd, yaml
from ctr.nbutils import setup, show
from ctr.bench import hardware_info

cfg, RES, FIG = setup()

Perfil: colab  |  resultados en: /content/deep-learning-01-ctr-mlp/results


In [2]:
pkgs = ["numpy", "pandas", "polars", "duckdb", "pyarrow", "scikit-learn", "scipy", "pyspark", "optuna", "lime",
        "shap", "matplotlib", "statsmodels", "threadpoolctl", "psutil", "jupyter-book"]
def v(p):
    try:
        return version(p)
    except PackageNotFoundError:
        return "no instalado"
try:
    java = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.strip().splitlines()[0]
except Exception as e:
    java = f"no disponible ({e})"
env = pd.DataFrame({"paquete": ["python", "java"] + pkgs, "version": [platform.python_version(), java] + [v(p) for p in pkgs]})
show(env, 30)
env.to_csv(RES / "versiones.csv", index=False)

,paquete,version
0,python,3.13.18
1,java,"openjdk version ""21.0.12.1"" 2026-08-18"
2,numpy,2.5.3
3,pandas,2.3.3
4,polars,2.1.0
5,duckdb,2.0.0
6,pyarrow,26.0.1
7,scikit-learn,1.9.1
8,scipy,1.18.1
9,pyspark,4.3.0


In [3]:
hw = hardware_info()
hw

{'sistema': 'Linux-6.6.159+-x86_64-with-glibc2.39',
 'procesador': 'x86_64',
 'nucleos_fisicos': 4,
 'nucleos_logicos': 8,
 'ram_gb': 51.0,
 'python': '3.13.18'}

In [4]:
print("Configuración del experimento (perfil:", cfg.profile, ")")
print(yaml.safe_dump(cfg.raw, allow_unicode=True, sort_keys=False))
print("Semillas de modelos:", cfg.seeds)
print("Muestreo: hash del id con semilla fija 1234 (muestras anidadas e idénticas en ambos entornos)")

Configuración del experimento (perfil: colab )
seed_base: 42
paths:
  raw_csv: data/raw/train.gz
  parquet_dir: data/parquet
  results_dir: results
  figures_dir: results/figures
split:
  train_days:
  - 21
  - 22
  - 23
  - 24
  - 25
  - 26
  - 27
  - 28
  valid_day: 29
  test_day: 30
features:
  high_card:
  - site_id
  - site_domain
  - app_id
  - app_domain
  - device_id
  - device_ip
  - device_model
  - C14
  - C17
  - C19
  - C20
  - C21
  low_card:
  - C1
  - banner_pos
  - site_category
  - app_category
  - device_type
  - device_conn_type
  - C15
  - C16
  - C18
  - hour_of_day
  - dow
  - franja
  crosses:
  - - site_id
    - device_type
  - - app_id
    - device_type
  - - banner_pos
    - device_conn_type
mlp_search_space:
  hidden_layer_sizes:
  - '50'
  - '100'
  - 100-50
  - 256-128
  alpha:
  - 1.0e-05
  - 0.1
  learning_rate_init:
  - 0.0001
  - 0.01
  batch_size:
  - 256
  - 512
  - 1024
  - 2048
spark_grid:
  hidden:
  - - 10
  - - 50
  - - 100
  - - 128
    - 64
  

In [5]:
from ctr import spark_ctr as S
spark = S.get_spark(cfg)
conf = pd.DataFrame(sorted(spark.sparkContext.getConf().getAll()), columns=["clave", "valor"])
conf = conf[conf.clave.str.match(r"spark\.(driver\.memory|master|sql\.shuffle|sql\.execution\.arrow|executor|app\.name|driver\.maxResultSize)")]
show(conf, 30)
spark.stop()

,clave,valor
1,spark.app.name,ctr-avazu
6,spark.driver.maxResultSize,4g
7,spark.driver.memory,28g
9,spark.executor.extraJavaOptions,-Djava.net.preferIPv6Addresses=false -XX:+Igno...
10,spark.executor.id,driver
11,spark.master,local[*]
14,spark.sql.execution.arrow.pyspark.enabled,true
15,spark.sql.shuffle.partitions,64


In [6]:
man = RES / "manifiesto_datos.json"
json.loads(man.read_text()) if man.exists() else "Ejecute el capítulo 1 para generar el manifiesto."

{'archivo': 'train.gz',
 'bytes': 1117527809,
 'sha256': '2884596ea9351c8a9abc02bff1f7a4f0d161eac8caceadaaccfff31ec638b4ca',
 'filas_por_dia': {'21': 4122995,
  '22': 5337126,
  '23': 3870752,
  '24': 3335302,
  '25': 3363122,
  '26': 3835892,
  '27': 3225010,
  '28': 5287222,
  '29': 3832608,
  '30': 4218938},
 'ingesta_s': 90.87243992200001}